# Ensemble de 2 modelos

## Modelo 9100   - BaseLine - Data Drifting "Deflacion" - 7 Semillas

- Se mantienen todos los parámetros del cuaderno base.

- Data Drifting: "deflacion" (predeterminado)

## Modelo 400000 - BaseLine - Data Drifting "Ninguno" - 7 Semillas

- Se mantienen todos los parámetros del cuaderno base.

- Data Drifting: "ninguno" (predeterminado)

- Como se contempla a último momento la posibilidad de ensamblar, se ejecuta este modelo con las semillas que el tiempo disponible permite. En el cuaderno se borra la etapa "A Kaggle" ya que sólo necesito el archivo "prediccion.txt".

## Modelo 100000 - BaseLine con recomendaciones - 14 Semillas

- Data Drifting: "estandarizacion"

- FE Histórico: lag3 - tag3

- Algoritmo Genético (recomendado por el grupo)

- Undersampling: 10%

Tomo los archivos de predicción, provenientes de ejecuciones con distintas semillas, con los que exploré cada experimento que voy a ensamblar.

Los archivos tienen todos el mismo nombre "prediccion.txt" en la respectiva carpeta que se crea para cada semilla. Antes de subirlos para su ensamble, renombrarlos con nombres:

- Modelo 9100: Desde "prediccion 100001.txt" hasta "prediccion 100007.txt".

- Modelo 400000: Desde "prediccion 100008.txt" hasta "prediccion 100014.txt".

- Modelo 100000: Desde "prediccion 100015.txt" hasta "prediccion 100028.txt".

**Importante 1:** se promedian probabilidades por cliente.

**Importante 2:** como en algunos casos los 7 cortes tuvieron máxima ganancia en uno u otro extremo, dado que es sencillo, extiendo en el ensamblado los cortes a 9, desde 1700 hasta 2500.

**Importante 3:** utilizo Chat GPT como ayuda para elaborar las partes de código que no conozco.

## Librerías y parámetros

La totalidad de archivos `prediccion XXXXXX.txt` deben estar previamente arrastrados a la **carpeta raíz de la sesión de Colab**.

Los números son únicamente identificadores correlativos que asigné a los archivos para facilitar su manipulación. **No representan las semillas utilizadas en los experimentos.** Las semillas correspondientes están en las carpetas que contienen el archivo con su nombre original, indicandose en la carpeta la semilla correspondiente.

Dado que compartiré los cuadernos con los que obtuve dichas predicciones, y en cada cuaderno se establece la semilla con la que se trabaja, se puede reproducir el proceso ejecutándose la totalidad de cuadernos para obtenerse al finalizar la totalidad de archivos con las predicciones que utilizo en este ensamble.

Recordar que en función de los comenarios anteriores, los archivos deben previo al ensamble renombrarse convenientemente.

Se generan nueve cortes, desde 1700 hasta 2500 cada 100 clientes.

In [1]:
library(data.table)

PARAM <- list()

PARAM$predicciones <- 100001:100028
PARAM$cortes <- seq(1700, 2500, 100)

PARAM


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%




$predicciones
 [1] 100001 100002 100003 100004 100005 100006 100007 100008 100009 100010
[11] 100011 100012 100013

$cortes
[1] 1700 1800 1900 2000 2100 2200 2300 2400 2500

## Verificación de los archivos en Colab

Primero observo los archivos disponibles en la carpeta de trabajo. Luego verifico automáticamente que estén presentes los 14 archivos esperados.

In [2]:
# Muestro los archivos disponibles en la carpeta de trabajo
list.files()

# Construyo los nombres esperados
archivos_esperados <- paste0("prediccion ", PARAM$predicciones, ".txt")

# Verifico si falta alguno
faltantes <- setdiff(archivos_esperados, list.files())

if (length(faltantes) > 0) {
  stop(
    "Faltan los siguientes archivos: ",
    paste(faltantes, collapse = ", ")
  )
}

cat("Se encontraron correctamente los",
    length(archivos_esperados),
    "archivos esperados.\n")

[1] "prediccion 100001.txt" "prediccion 100002.txt" "prediccion 100003.txt"
 [4] "prediccion 100004.txt" "prediccion 100005.txt" "prediccion 100006.txt"
 [7] "prediccion 100007.txt" "prediccion 100008.txt" "prediccion 100009.txt"
[10] "prediccion 100010.txt" "prediccion 100011.txt" "prediccion 100012.txt"
[13] "prediccion 100013.txt" "sample_data"

Se encontraron correctamente los 13 archivos esperados.


## Lectura de las 14 predicciones

Cada archivo contiene el identificador del cliente (`numero_de_cliente`) y la probabilidad (`prob`) calculada por una semilla.

Al leer cada archivo renombro la columna `prob` incorporando el número de semilla. De esta manera puedo conservar e identificar las 14 probabilidades antes de calcular el promedio.

In [3]:
lista_predicciones <- vector("list", length(PARAM$predicciones))
names(lista_predicciones) <- as.character(PARAM$predicciones)

for (i in seq_along(PARAM$predicciones)) {

  numero_prediccion <- PARAM$predicciones[i]
  archivo <- paste0("prediccion ", numero_prediccion, ".txt")

  tb <- fread(archivo)

  # Verifico la estructura esperada
  if (!identical(names(tb), c("numero_de_cliente", "prob"))) {
    stop("Estructura inesperada en: ", archivo)
  }

  # Identifico la probabilidad con su archivo de origen
  setnames(tb, "prob", paste0("prob_", numero_prediccion))

  lista_predicciones[[i]] <- tb
}

cat("Lectura de las 14 predicciones finalizada.\n")

Lectura de las 14 predicciones finalizada.


## Revisión de los archivos de entrada

Por las dudas de que no hubiese alguna corrupción de los archivos creados y descargados, realizo una serie de chequeos para verificar que son todos compatibles entre sí antes del ensamble.

Esto permite comprobar que los archivos son compatibles antes de realizar el promedio.

In [4]:
auditoria <- rbindlist(
  lapply(seq_along(lista_predicciones), function(i) {

    tb <- lista_predicciones[[i]]
    columna_prob <- names(tb)[2]

    data.table(
      prediccion = PARAM$predicciones[i],
      registros = nrow(tb),
      clientes_unicos = uniqueN(tb$numero_de_cliente),
      duplicados = sum(duplicated(tb$numero_de_cliente)),
      prob_NA = sum(is.na(tb[[columna_prob]]))
    )
  })
)

auditoria

prediccion,registros,clientes_unicos,duplicados,prob_NA
<int>,<int>,<int>,<int>,<int>
100001,33080,33080,0,0
100002,33080,33080,0,0
100003,33080,33080,0,0
100004,33080,33080,0,0
100005,33080,33080,0,0
100006,33080,33080,0,0
100007,33080,33080,0,0
100008,33080,33080,0,0
100009,33080,33080,0,0


## Verificación del conjunto de clientes

No asumo que las filas de los 14 archivos estén en el mismo orden.

Compruebo que todos contengan exactamente el mismo conjunto de `numero_de_cliente`. Posteriormente, las predicciones se unirán utilizando este identificador.

In [5]:
clientes_referencia <- sort(lista_predicciones[[1]]$numero_de_cliente)

mismos_clientes <- sapply(
  lista_predicciones,
  function(tb) {
    identical(
      sort(tb$numero_de_cliente),
      clientes_referencia
    )
  }
)

print(mismos_clientes)

if (!all(mismos_clientes)) {
  stop("No todos los archivos contienen exactamente los mismos clientes.")
}

cat("Los 14 archivos contienen el mismo conjunto de clientes.\n")

100001 100002 100003 100004 100005 100006 100007 100008 100009 100010 100011 
  TRUE   TRUE   TRUE   TRUE   TRUE   TRUE   TRUE   TRUE   TRUE   TRUE   TRUE 
100012 100013 
  TRUE   TRUE 
Los 14 archivos contienen el mismo conjunto de clientes.


## Unión de las 14 predicciones por cliente

Uno las tablas utilizando `numero_de_cliente` como clave.

De esta forma, cada fila reúne las 14 probabilidades correspondientes al mismo cliente independientemente del orden original de los archivos.

In [6]:
tb_ensemble <- Reduce(
  function(x, y) {
    merge(
      x,
      y,
      by = "numero_de_cliente",
      all = FALSE,
      sort = FALSE
    )
  },
  lista_predicciones
)

cat("Cantidad de clientes:", nrow(tb_ensemble), "\n")
cat("Cantidad de columnas:", ncol(tb_ensemble), "\n")

tb_ensemble[1:10]

Cantidad de clientes: 33080 
Cantidad de columnas: 14 


numero_de_cliente,prob_100001,prob_100002,prob_100003,prob_100004,prob_100005,prob_100006,prob_100007,prob_100008,prob_100009,prob_100010,prob_100011,prob_100012,prob_100013
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
11675570,0.0024342920,0.002695111,0.0020148875,0.0012816629,0.0021627593,0.0019727405,0.0021464970,0.0019618325,0.0025147245,0.0022239035,0.0023603866,0.0018252051,0.002285180
11675731,0.0010659157,0.001994326,0.0011825367,0.0009950632,0.0012048228,0.0011968435,0.0013631656,0.0013612560,0.0016818616,0.0010242987,0.0012698509,0.0011435461,0.001745444
11677630,0.0059266168,0.006628468,0.0069529886,0.0063963171,0.0055042897,0.0054969570,0.0062005395,0.0062392082,0.0050407542,0.0041847737,0.0049522534,0.0047858639,0.007470351
11678491,0.0014994892,0.002376822,0.0015983119,0.0011609002,0.0012546855,0.0012715839,0.0017692908,0.0014498406,0.0016648087,0.0008596861,0.0009752895,0.0011160415,0.002203787
11678621,0.0014555791,0.002438384,0.0016456307,0.0012244125,0.0012854493,0.0012871606,0.0018110280,0.0014743721,0.0021766524,0.0012826270,0.0026759374,0.0011026969,0.002200138
11679331,0.0013465164,0.001903467,0.0015968692,0.0025057804,0.0014881219,0.0030574941,0.0024482339,0.0012078653,0.0015274045,0.0021413104,0.0013563941,0.0035997993,0.001903596
11680281,0.0011186973,0.001954264,0.0013702128,0.0011542076,0.0012541113,0.0012008339,0.0014110492,0.0016221535,0.0014622722,0.0007891395,0.0010127421,0.0012223368,0.001524388
11682860,0.0007735494,0.001552374,0.0008553604,0.0008459424,0.0008790723,0.0006327323,0.0009552043,0.0006288063,0.0007872655,0.0005499604,0.0004868140,0.0004767977,0.001183223
11683331,0.0103066094,0.010980528,0.0163813272,0.0168455766,0.0137933121,0.0161872910,0.0113357510,0.0096197214,0.0094476375,0.0108760912,0.0077469019,0.0115675706,0.010771445


## Cálculo de la probabilidad promedio

Para cada cliente calculo el promedio simple de las 14 probabilidades.

In [7]:
columnas_prob <- grep(
  "^prob_",
  names(tb_ensemble),
  value = TRUE
)

tb_ensemble[, prob := rowMeans(.SD), .SDcols = columnas_prob]

tb_promedio <- tb_ensemble[, .(
  numero_de_cliente,
  prob
)]

summary(tb_promedio$prob)

    Min.  1st Qu.   Median     Mean  3rd Qu.     Max. 
0.000734 0.000955 0.001842 0.011813 0.004610 0.663297 

## Archivo de probabilidades del ensemble

Guardo la probabilidad promedio de cada cliente antes de aplicar cualquier corte.

In [8]:
archivo_promedio <- "prediccion_ensemble_28_predicciones.txt"

fwrite(
  tb_promedio,
  file = archivo_promedio,
  sep = "\t"
)

cat("Archivo generado:", archivo_promedio, "\n")

Archivo generado: prediccion_ensemble_14_predicciones.txt 


## Construcción del ranking

Ordeno los clientes desde la mayor hasta la menor probabilidad promedio.

En caso de existir un empate exacto de probabilidades, utilizo `numero_de_cliente` en orden ascendente como segundo criterio para que el procedimiento sea completamente reproducible.

In [9]:
tb_ranking <- copy(tb_promedio)

setorder(
  tb_ranking,
  -prob,
  numero_de_cliente
)

tb_ranking[, posicion := .I]

tb_ranking[1:20]

numero_de_cliente,prob,posicion
<int>,<dbl>,<int>
11893651,0.6632972,1
36662290,0.6593808,2
20196060,0.6456878,3
34080250,0.6409224,4
41547330,0.6193873,5
30288530,0.6187292,6
45275050,0.6154531,7
54552140,0.6120729,8
29442550,0.6081903,9


## Generación de los nueve cortes para Kaggle

Para cada corte:

- los primeros `N` clientes del ranking reciben `Predicted = 1`;
- los restantes clientes reciben `Predicted = 0`.

Genero los cortes 1700, 1800, 1900, 2000, 2100, 2200, 2300, 2400 y 2500.

In [10]:
archivos_generados <- character()

for (corte in PARAM$cortes) {

  submission <- tb_ranking[, .(
    numero_de_cliente,
    Predicted = as.integer(posicion <= corte)
  )]

  archivo_salida <- paste0(
    "KA_ensemble28_",
    corte,
    ".csv"
  )

  fwrite(
    submission,
    file = archivo_salida,
    sep = ","
  )

  archivos_generados <- c(
    archivos_generados,
    archivo_salida
  )
}

archivos_generados

[1] "KA_ensemble14_1700.csv" "KA_ensemble14_1800.csv" "KA_ensemble14_1900.csv"
[4] "KA_ensemble14_2000.csv" "KA_ensemble14_2100.csv" "KA_ensemble14_2200.csv"
[7] "KA_ensemble14_2300.csv" "KA_ensemble14_2400.csv" "KA_ensemble14_2500.csv"